In [1]:
!pip install pandas scikit-learn nltk openpyxl

import pandas as pd
import numpy as np
import re
import string
import nltk

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
from google.colab import files
uploaded = files.upload()

Saving phishing_email_dataset2.xlsx to phishing_email_dataset2.xlsx


In [4]:
df = pd.read_excel('phishing_email_dataset2.xlsx')

# Combine Subject + Body into one text column
df['text'] = df['Subject'] + " " + df['Body']

# Convert Label (Phishing/Safe) → numeric
df['label'] = df['Label'].map({'Phishing': 1, 'Safe': 0})

# Keep useful columns
df = df[['text', 'URL', 'Contains_Suspicious_Keywords',
         'URL_Length', 'Has_IP_in_URL',
         'Num_Links', 'Has_Attachment', 'label']]

print(df.head())

                                                text  \
0  Urgent action required Click here to update pa...   
1  Meeting scheduled for tomorrow Invoice attache...   
2  Invoice attached for your review Your order ha...   
3  Urgent action required Urgent action required ...   
4  Your account will be suspended Verify your acc...   

                                       URL  Contains_Suspicious_Keywords  \
0  https://secure-login-alert.com/mzxcowub                             1   
1                https://example.com/nnsaj                             0   
2                https://service.org/lgyxs                             0   
3  https://verify-account-now.net/dydgbsad                             1   
4       http://security-check.xyz/oymukolk                             1   

   URL_Length  Has_IP_in_URL  Num_Links  Has_Attachment  label  
0          71              1          1               1      1  
1          26              0          2               1      0  
2          

In [5]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " URL ", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_text'] = df['text'].apply(preprocess_text)

In [6]:
# Text features
vectorizer = TfidfVectorizer(max_features=3000)
X_text = vectorizer.fit_transform(df['text']).toarray()

# Structured features (already in dataset)
X_extra = df[['Contains_Suspicious_Keywords',
              'URL_Length',
              'Has_IP_in_URL',
              'Num_Links',
              'Has_Attachment']].values

# Combine all features
import numpy as np
X = np.hstack((X_text, X_extra))

y = df['label']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=300, random_state=42)

In [9]:
y_pred = rf.predict(X_test)

In [10]:
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 1.0

Confusion Matrix:
 [[15  0]
 [ 0 15]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       1.00      1.00      1.00        15

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

